In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np

# === Load feature-engineered data ===
df_with = pd.read_csv("feature_engineered_posts.csv")

# === Define features and target ===
feature_cols = [
    'caption_length_words',
    'caption_length_chars',
    'month',
    'hour',
    'clicks',
    'impressions',
    'click_through_rate_(ctr)',
    'engagement_rate'
]

# Encode categorical columns
df_with = pd.get_dummies(df_with, columns=['post_type', 'day_of_week'], drop_first=True)

# Final feature list after dummy encoding
X = df_with[feature_cols + [col for col in df_with.columns if col.startswith('post_type_') or col.startswith('day_of_week_')]]
y = df_with['total_engagement']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train LightGBM model
model = LGBMRegressor(random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print(f" Model trained. RMSE: {rmse:.2f}, R²: {r2:.2f}")

# === Save model ===
joblib.dump(model, "post_recommender_model.pkl")
print("Model saved as post_recommender_model.pkl")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000028 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 107
[LightGBM] [Info] Number of data points in the train set: 47, number of used features: 7
[LightGBM] [Info] Start training from score 25.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [7]:
import pandas as pd
import joblib
import numpy as np

# === Load trained LightGBM model ===
model = joblib.load("post_recommender_model.pkl")

# === Get exact feature names the model was trained on ===
model_features = model.feature_name_
print("Model expects these features:")
print(model_features)

# === Define 3 sample post ideas (video, text, image) ===
new_posts = pd.DataFrame([
    {
        "caption_length_words": 25,
        "caption_length_chars": 150,
        "month": 7,
        "hour": 9,
        "clicks": 40,
        "impressions": 2500,
        "click_through_rate_(ctr)": 1.6,
        "engagement_rate": 3.8,
        "post_type_video": 1,
        "post_type_text": 0,
        "post_type_image": 0,
        "day_of_week_Monday": 1,
        "day_of_week_Tuesday": 0,
        "day_of_week_Wednesday": 0,
        "day_of_week_Thursday": 0,
        "day_of_week_Friday": 0,
        "day_of_week_Saturday": 0,
        "day_of_week_Sunday": 0
    },
    {
        "caption_length_words": 12,
        "caption_length_chars": 70,
        "month": 7,
        "hour": 17,
        "clicks": 20,
        "impressions": 1800,
        "click_through_rate_(ctr)": 1.2,
        "engagement_rate": 2.9,
        "post_type_video": 0,
        "post_type_text": 1,
        "post_type_image": 0,
        "day_of_week_Monday": 0,
        "day_of_week_Tuesday": 1,
        "day_of_week_Wednesday": 0,
        "day_of_week_Thursday": 0,
        "day_of_week_Friday": 0,
        "day_of_week_Saturday": 0,
        "day_of_week_Sunday": 0
    },
    {
        "caption_length_words": 15,
        "caption_length_chars": 90,
        "month": 7,
        "hour": 14,
        "clicks": 30,
        "impressions": 2000,
        "click_through_rate_(ctr)": 1.4,
        "engagement_rate": 3.2,
        "post_type_video": 0,
        "post_type_text": 0,
        "post_type_image": 1,
        "day_of_week_Monday": 0,
        "day_of_week_Tuesday": 0,
        "day_of_week_Wednesday": 0,
        "day_of_week_Thursday": 1,
        "day_of_week_Friday": 0,
        "day_of_week_Saturday": 0,
        "day_of_week_Sunday": 0
    }
])

# === Align columns with model features ===
new_posts = new_posts[model_features]

# === Predict engagement ===
new_posts["predicted_engagement"] = model.predict(new_posts)

# === Rank and recommend ===
recommendations = new_posts.sort_values("predicted_engagement", ascending=False)
print("Top recommended post ideas:")
print(recommendations[["predicted_engagement"]])

# === Save to CSV ===
recommendations.to_csv("recommended_post_ideas.csv", index=False)
print("Recommendations saved to recommended_post_ideas.csv")


✅ Model expects these features:
['caption_length_words', 'caption_length_chars', 'month', 'hour', 'clicks', 'impressions', 'click_through_rate_(ctr)', 'engagement_rate', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']
📢 Top recommended post ideas:
   predicted_engagement
0             38.595131
2             38.595131
1             28.096858
✅ Recommendations saved to recommended_post_ideas.csv
